# Orbit: distill the team, train it, score it once

This notebook runs the existing team-distill path end to end on a free Colab T4:

1. **Teacher dialogs.** `finetune/distill_teacher.py` asks a Claude model to play planner, worker, and critic on train and tune tasks only. Every tool call runs in Orbit's sandbox. A dialog is kept only when the task checker passes and every turn matches its role schema. The frozen test is a local blocklist and is never sent.
2. **Student.** `finetune/train.py` trains Qwen2.5-1.5B-Instruct with QLoRA using `finetune/configs/qlora_distill.json`, unchanged.
3. **One frozen score.** `python -m orbit_agents.eval.run` scores `team_distill` once on `eval/test.jsonl` (protocol `team-eval-v1`) under harness v3, into `results/v3_qwen1.5b/team_distill.json`.
4. **Baselines and README.** The v3 baselines a fair comparison needs (single base, team base, team plus the medium adapter) are scored once each if their files are not already there. Then `orbit_agents.eval.compare` writes the paired bootstrap, and `orbit_agents.eval.readme` rewrites the README results block from those files.

Rules this notebook keeps:

- Do not score the frozen test in a loop. A scoring cell refuses to overwrite a measured file.
- The API key comes from Colab secrets (or the environment). It is never printed or written to disk.
- The repo's hard caps stay on: 800 requests per run and a 5 USD estimated-cost stop for Anthropic.
- Iterate on the tune split with `python -m orbit_agents.eval.run --split tune`, not on the frozen test.
- This notebook does not run the frontier API team as a scored policy. The repo has no such eval path.

Wall clock on a T4 has not been measured for this notebook. Before you start, add a Colab secret named `ORBIT_ANTHROPIC_API_KEY` (key icon in the left sidebar) and give this notebook access to it. Then pick Runtime > Change runtime type > T4 GPU and run all cells.

## Parameters

In [ ]:
# Teacher. claude-haiku-5-5 is the repo default. claude-sonnet-5-5 is in the repo's price
# table and is an option for higher-quality dialogs at a higher cost per token.
TEACHER_MODEL = 'claude-haiku-5-5'
TEACHER_PROVIDER = 'anthropic'        # the repo also supports 'gemini' (key: ORBIT_TEACHER_API_KEY)

# Hard caps. These are the repo defaults from finetune/distill_teacher.py.
MAX_REQUESTS = 800                    # cap for one teacher run
MAX_COST_USD = 5.0                    # Anthropic only; the estimate is cumulative across resumed runs
REQUESTS_PER_MINUTE = None            # None keeps the repo pace (20 per minute for Anthropic)
ATTEMPTS_PER_TASK = 3
LIMIT_TASKS = None                    # e.g. 5 for a short smoke run of the teacher; None uses the full pool

# Frozen-test scoring under harness v3.
SCORE_V3_BASELINES = True             # single_base and team_base, once each, if their v3 files are missing
TRAIN_MEDIUM_IF_MISSING = True        # train qlora_medium.json once so team_lora can be scored under v3
MEDIUM_ADAPTER = 'finetune/outputs/adapter'

REPO_URL = 'https://github.com/shmorsi/orbit-multi-agent'
MOUNT_DRIVE = False                   # True copies outputs to /content/drive/MyDrive/orbit-distill as well

## Project and install

Upload a project zip to `/content/orbit.zip` to run that copy, for example one that already holds `finetune/data/distill/` from an earlier teacher run. With no zip, the cell clones `REPO_URL`. The install matches `finetune/colab_train.ipynb`: it uninstalls torchao (Colab's build breaks `PeftModel.from_pretrained`), installs `finetune/requirements-colab.txt` without replacing Colab's CUDA torch, and runs the leakage check.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if not Path('pyproject.toml').exists():
    zip_path = Path('/content/orbit.zip')
    if zip_path.exists():
        dest = Path('/content/orbit')
        if dest.exists():
            shutil.rmtree(dest)
        dest.mkdir(parents=True)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(dest)
        roots = [dest] if (dest / 'pyproject.toml').exists() else [p for p in dest.iterdir() if (p / 'pyproject.toml').exists()]
        if len(roots) != 1:
            raise SystemExit('The zip does not contain pyproject.toml at its top level.')
        os.chdir(roots[0])
        print('Using the uploaded project.')
    else:
        target = Path('/content/orbit-repo')
        if target.exists():
            shutil.rmtree(target)
        subprocess.check_call(['git', 'clone', REPO_URL, str(target)])
        os.chdir(target)
        print('Cloned', REPO_URL)
print('project', Path.cwd())

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'No GPU visible. Training and scoring need a CUDA runtime (Runtime > Change runtime type > T4 GPU).')

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed', '-r', 'finetune/requirements-colab.txt'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
# Import check in a subprocess: Colab's kernel puts /content on sys.path and can load a folder as a namespace package.
print(subprocess.check_output([sys.executable, '-c', 'import orbit_agents, os; print(orbit_agents.__version__, os.path.dirname(orbit_agents.__file__))'], text=True))

ENV = os.environ.copy()
ENV['PYTHONUNBUFFERED'] = '1'
ENV['PYTHONPATH'] = os.pathsep.join(part for part in (str(Path.cwd() / 'src'), ENV.get('PYTHONPATH', '')) if part)
subprocess.check_call([sys.executable, 'finetune/leakage.py'], env=ENV)

def run(command, env=None):
    print(' '.join(str(part) for part in command), flush=True)
    return subprocess.run([str(part) for part in command], env=env or ENV)

def check(command, env=None):
    result = run(command, env)
    if result.returncode != 0:
        raise SystemExit(f'command failed with exit code {result.returncode}')

## Protocol checks

The frozen test must match the hash recorded in `eval/manifest.json`, and the distill config must be the committed one. The cell also lists which v3 result files already exist, so nothing is scored twice.

In [ ]:
import hashlib, json

manifest = json.loads(Path('eval/manifest.json').read_text())
digest = hashlib.sha256(Path('eval/test.jsonl').read_bytes()).hexdigest()
if manifest['protocol'] != 'team-eval-v1' or digest != manifest['test_sha256']:
    raise SystemExit('eval/test.jsonl does not match eval/manifest.json. Do not score a changed frozen test.')
print('frozen test', manifest['protocol'], 'hash matches the manifest')

DISTILL_CONFIG = 'finetune/configs/qlora_distill.json'
MEDIUM_CONFIG = 'finetune/configs/qlora_medium.json'
config = json.loads(Path(DISTILL_CONFIG).read_text())
assert config['base_model'] == 'Qwen/Qwen2.5-1.5B-Instruct'
assert config['method'] == 'qlora'
assert config['train_file'] == 'finetune/data/distill/train.jsonl'
assert config['tune_file'] == 'finetune/data/distill/tune.jsonl'
assert MAX_REQUESTS <= config['teacher_max_requests'], 'MAX_REQUESTS may not exceed the repo cap'
if TEACHER_PROVIDER == 'anthropic':
    assert MAX_COST_USD <= config['anthropic_max_cost_usd'], 'MAX_COST_USD may not exceed the repo limit'
BASE_MODEL = config['base_model']
print('distill config', DISTILL_CONFIG, BASE_MODEL, config['method'], 'rank', config['lora_r'])

V3 = Path('results/v3_qwen1.5b')
V3.mkdir(parents=True, exist_ok=True)

def measured(path):
    path = Path(path)
    if not path.is_file():
        return False
    payload = json.loads(path.read_text())
    return payload.get('status') == 'measured' and payload.get('kind') == 'model_eval'

for name in ('single_base', 'team_base', 'team_lora', 'team_distill'):
    path = V3 / f'{name}.json'
    print(f'{path}:', 'measured' if measured(path) else 'not scored')
if measured(V3 / 'team_distill.json'):
    raise SystemExit('results/v3_qwen1.5b/team_distill.json is already measured. The frozen test is scored once per model.')

## Teacher pool and request estimate

`--dry-run` builds the task pool and drops anything near the frozen test. It sends no request and reads no key. The estimate below is an upper bound: each attempt is one team run of at most `MAX_TURNS` model calls, plus one preflight. Retries after a server error or a rate limit also count toward the cap and are not in the bound. The hard cap stops the run first if the bound is higher. A capped run keeps its dialogs and writes a checkpoint, so rerunning the teacher cell resumes where it stopped.

In [ ]:
DISTILL_DIR = Path('finetune/data/distill')
teacher_args = ['--provider', TEACHER_PROVIDER, '--model', TEACHER_MODEL, '--attempts', ATTEMPTS_PER_TASK]
if LIMIT_TASKS:
    teacher_args += ['--limit-tasks', LIMIT_TASKS]

dry = subprocess.run([sys.executable, 'finetune/distill_teacher.py', '--dry-run', *map(str, teacher_args)], env=ENV, capture_output=True, text=True)
if dry.returncode != 0:
    print(dry.stderr)
    raise SystemExit('distill dry run failed')
pool = json.loads(dry.stdout)
max_turns = int(subprocess.check_output([sys.executable, '-c', 'from orbit_agents.engine import MAX_TURNS; print(MAX_TURNS)'], env=ENV, text=True))
n_tasks = pool['n_train_tasks'] + pool['n_tune_tasks']
bound = 1 + n_tasks * ATTEMPTS_PER_TASK * max_turns
print(json.dumps({key: pool[key] for key in ('provider', 'model', 'n_train_tasks', 'n_tune_tasks', 'n_refusals', 'n_dropped', 'requests_per_minute')}, indent=2))
print(f'Upper bound: 1 preflight + {n_tasks} tasks x {ATTEMPTS_PER_TASK} attempts x {max_turns} calls = {bound} requests.')
print(f'This run stops at {min(bound, MAX_REQUESTS)} requests (hard cap {MAX_REQUESTS}).')
if TEACHER_PROVIDER == 'anthropic':
    print(f'It also stops when the estimated cost reaches {MAX_COST_USD:g} USD.')

## API key

The key is read from the Colab secret `ORBIT_ANTHROPIC_API_KEY` (or `ORBIT_TEACHER_API_KEY` for Gemini), falling back to an environment variable of the same name. It is passed only to the teacher process and is never printed.

In [ ]:
KEY_NAME = 'ORBIT_ANTHROPIC_API_KEY' if TEACHER_PROVIDER == 'anthropic' else 'ORBIT_TEACHER_API_KEY'
_key = ''
try:
    from google.colab import userdata
    _key = (userdata.get(KEY_NAME) or '').strip()
except ImportError:
    pass
except Exception as exc:
    print(f'Colab secret {KEY_NAME} not readable ({type(exc).__name__}). Trying the environment.')
if not _key:
    _key = os.environ.get(KEY_NAME, '').strip()
if not _key:
    raise SystemExit(
        f'No API key. Add a Colab secret named {KEY_NAME} (key icon in the left sidebar), '
        'turn on notebook access for it, and rerun this cell.'
    )
for name in ('ORBIT_ANTHROPIC_API_KEY', 'ORBIT_TEACHER_API_KEY'):
    ENV.pop(name, None)  # only the teacher process gets a key
TEACHER_ENV = dict(ENV)
TEACHER_ENV[KEY_NAME] = _key
del _key
print(f'{KEY_NAME} found. It is passed to the teacher process only.')

## Teacher run

Exit code 2 means the run stopped at a quota, billing, or cost limit. The dialogs kept so far are on disk, `stats.json` records why it stopped, and rerunning this cell resumes from `finetune/data/distill/checkpoint.json`. Exit code 1 means an HTTP, network, or leakage stop: nothing is trained in that case.

In [ ]:
command = [
    sys.executable, 'finetune/distill_teacher.py', *teacher_args,
    '--max-requests', MAX_REQUESTS,
    '--max-cost-usd', MAX_COST_USD,
    '--output', DISTILL_DIR,
]
if REQUESTS_PER_MINUTE:
    command += ['--requests-per-minute', REQUESTS_PER_MINUTE]
result = run(command, env=TEACHER_ENV)
if result.returncode == 1:
    raise SystemExit('The teacher stopped on an HTTP, network, or leakage error. Fix that before training.')
if result.returncode not in (0, 2):
    raise SystemExit(f'teacher exited with {result.returncode}')
stats = json.loads((DISTILL_DIR / 'stats.json').read_text())
print(json.dumps({key: stats.get(key) for key in ('provider', 'teacher_model', 'requests', 'input_tokens', 'output_tokens', 'estimated_cost_usd', 'n_train_dialogs', 'n_tune_dialogs', 'stopped_reason')}, indent=2))
print((DISTILL_DIR / 'manifest.json').read_text())

## Train the student

The dry run re-checks the distilled rows against the frozen test before any GPU time is spent. Training uses `finetune/configs/qlora_distill.json` as committed and writes `finetune/outputs-distill/`.

In [ ]:
check([sys.executable, 'finetune/train.py', '--config', DISTILL_CONFIG, '--dry-run'])
check([sys.executable, 'finetune/train.py', '--config', DISTILL_CONFIG, '--output', 'finetune/outputs-distill'])
DISTILL_ADAPTER = Path('finetune/outputs-distill/adapter')
if not DISTILL_ADAPTER.exists():
    raise SystemExit('Training did not write an adapter. There is no score to invent.')
print(Path('finetune/outputs-distill/RUN.json').read_text())
curve = Path('finetune/outputs-distill/loss_curve.png')
if curve.exists():
    from IPython.display import Image, display
    display(Image(filename=str(curve)))

## Score team distill once

One frozen score with the repo's scorer. The cell refuses to run if the file is already measured.

In [ ]:
def score_once(mode, out, adapter=None):
    out = Path(out)
    if measured(out):
        print(f'{out} is already measured. Not scoring the frozen test again.')
        return
    command = [
        sys.executable, '-m', 'orbit_agents.eval.run',
        '--split', 'test',
        '--mode', mode,
        '--backend', 'transformers',
        '--model', BASE_MODEL,
        '--out', out,
    ]
    if adapter:
        command += ['--adapter', adapter]
    check(command)
    print(out.read_text()[:400])

score_once('team', V3 / 'team_distill.json', adapter=DISTILL_ADAPTER)

## Harness v3 baselines

The v2 numbers were measured before the harness changed, so they are not compared with this score. A v3 comparison needs single base, team base, and team plus the medium adapter under v3. Each is scored once and skipped if its file already exists. If `MEDIUM_ADAPTER` is missing and `TRAIN_MEDIUM_IF_MISSING` is on, `finetune/configs/qlora_medium.json` is trained once first. That is a retrain with the same config and seed, not the v2 adapter file. To score the v2 adapter itself, put it at `finetune/outputs/adapter` in the uploaded zip.

If the `ORBIT_RESCORE` cell in `finetune/colab_train.ipynb` already wrote these files, they are found and not scored again.

In [ ]:
if SCORE_V3_BASELINES:
    score_once('single', V3 / 'single_base.json')
    score_once('team', V3 / 'team_base.json')
medium = Path(MEDIUM_ADAPTER)
if not measured(V3 / 'team_lora.json'):
    if not medium.exists() and TRAIN_MEDIUM_IF_MISSING:
        check([sys.executable, 'finetune/train.py', '--config', MEDIUM_CONFIG, '--output', medium.parent])
    if medium.exists():
        score_once('team', V3 / 'team_lora.json', adapter=medium)
    else:
        print(f'No adapter at {medium}. team_lora stays not scored under v3.')

## Comparison and README

The paired bootstrap runs only when all four v3 files are measured. The README block is then rewritten from the results files by `python -m orbit_agents.eval.readme`, the same generator the test suite checks.

In [ ]:
needed = [V3 / f'{name}.json' for name in ('single_base', 'team_base', 'team_lora', 'team_distill')]
if all(measured(path) for path in needed):
    check([
        sys.executable, '-m', 'orbit_agents.eval.compare',
        '--single', needed[0], '--team', needed[1], '--lora', needed[2], '--distill', needed[3],
        '--out', V3 / 'comparison.json',
    ])
else:
    print('Not every v3 file is measured, so no paired comparison is written.')
check([sys.executable, '-m', 'orbit_agents.eval.readme'])
check([sys.executable, '-m', 'orbit_agents.eval.readme', '--check'])
text = Path('README.md').read_text()
start, end = text.index('<!-- RESULTS_START -->'), text.index('<!-- RESULTS_END -->')
print(text[start:end])

## Download

The zip holds what belongs in the repo (`results/v3_qwen1.5b/*.json`, `README.md`) and what stays out of git (adapters, distilled dialogs, the teacher checkpoint). Commit only the results files and the README.

In [ ]:
bundle = Path('/content/orbit-distill-outputs')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir(parents=True)
items = [Path('README.md'), V3, Path('finetune/outputs-distill'), DISTILL_DIR]
if medium.exists():
    items.append(medium.parent)
for path in items:
    if not path.exists():
        continue
    target = bundle / path
    target.parent.mkdir(parents=True, exist_ok=True)
    if path.is_dir():
        shutil.copytree(path, target, ignore=shutil.ignore_patterns('checkpoint-*'))
    else:
        shutil.copy2(path, target)
archive = shutil.make_archive('/content/orbit-distill-outputs', 'zip', bundle)
print(archive)
if MOUNT_DRIVE and Path('/content/drive/MyDrive').exists():
    drive_dir = Path('/content/drive/MyDrive/orbit-distill')
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, drive_dir)
    print('copied to', drive_dir)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print('Download skipped outside Colab:', type(exc).__name__)

Do not score the frozen test in a loop. A new model gets one frozen score; iterate on `--split tune`.